In [15]:
#IAAA
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import math

print("Libraries loaded.")
print("MPS available:", torch.backends.mps.is_available())

Libraries loaded.
MPS available: True


In [16]:
X_train = np.load('../data/processed/X_train_resampled.npy')
y_train = pd.read_csv('../data/processed/y_train_resampled.csv').squeeze()
X_val = np.load('../data/processed/X_val_scaled.npy')
y_val = pd.read_csv('../data/processed/y_val_clean.csv').squeeze()

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)

X_train shape: (2062829, 71)
X_val shape: (423051, 71)


In [17]:
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_val_encoded = le.transform(y_val)



In [18]:
#defining device
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

Device: mps


In [19]:
# Load original y_train before SMOTE for true class frequencies
y_train_original = pd.read_csv('../data/processed/y_train_clean.csv').squeeze()

total_original = len(y_train_original)
class_freq_original = {}
for label in le.classes_:
    count = (y_train_original == label).sum()
    class_freq_original[label] = count / total_original

# Build frequency bias tensor from original frequencies
freq_bias = torch.zeros(len(le.classes_))
for i, label in enumerate(le.classes_):
    freq_bias[i] = math.log(1.0 / class_freq_original[label])

freq_bias = freq_bias.to(device)

print("Original frequency bias:")
for i, label in enumerate(le.classes_):
    print(f"  {label}: {freq_bias[i]:.4f}")

Original frequency bias:
  BENIGN: 0.2192
  Bot: 7.2763
  DDoS: 3.0950
  DoS GoldenEye: 5.6158
  DoS Hulk: 2.5086
  DoS Slowhttptest: 6.2428
  DoS slowloris: 6.1900
  FTP-Patator: 5.8759
  Heartbleed: 12.5530
  Infiltration: 11.2408
  PortScan: 2.8796
  SSH-Patator: 6.1729
  Web Attack - Brute Force: 7.5367
  Web Attack - Sql Injection: 11.7908
  Web Attack - XSS: 8.3742


In [21]:
# Class weights using log inverse frequency — same as logit adjustment formula
class_weights = torch.zeros(len(le.classes_))
for i, label in enumerate(le.classes_):
    class_weights[i] = math.log(1.0 / class_freq_original[label])

# Normalize so weights sum to num_classes
class_weights = class_weights / class_weights.sum() * len(le.classes_)
class_weights = class_weights.to(device)

print("Class weights:")
for i, label in enumerate(le.classes_):
    print(f"  {label}: {class_weights[i]:.4f}")

Class weights:
  BENIGN: 0.0337
  Bot: 1.1186
  DDoS: 0.4758
  DoS GoldenEye: 0.8633
  DoS Hulk: 0.3857
  DoS Slowhttptest: 0.9597
  DoS slowloris: 0.9516
  FTP-Patator: 0.9033
  Heartbleed: 1.9298
  Infiltration: 1.7281
  PortScan: 0.4427
  SSH-Patator: 0.9490
  Web Attack - Brute Force: 1.1586
  Web Attack - Sql Injection: 1.8126
  Web Attack - XSS: 1.2874


In [22]:
class NetworkFlowDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(NetworkFlowDataset(X_train, y_train_encoded), batch_size=512, shuffle=True)
val_loader = DataLoader(NetworkFlowDataset(X_val, y_val_encoded), batch_size=512, shuffle=False)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")


Training batches: 4029
Validation batches: 827


In [23]:
class IAAAttention(nn.Module):
    def __init__(self, d_model, nhead):
        super(IAAAttention, self).__init__()
        
        self.d_model = d_model
        self.nhead = nhead
        self.d_head = d_model // nhead
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
            
    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        
        Q = Q.view(batch_size, seq_len, self.nhead, self.d_head).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.nhead, self.d_head).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.nhead, self.d_head).transpose(1, 2)
        
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_head)
        
        
        weights = F.softmax(scores, dim=-1)
        output = torch.matmul(weights, V)
        
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        output = self.W_o(output)
        
        return output

In [24]:
class IAATransformerLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=512, dropout=0.3):
        super(IAATransformerLayer, self).__init__()
        
        self.attention = IAAAttention(d_model, nhead)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # Attention + residual connection
        attn_output = self.attention(x)
        x = self.norm1(x + self.dropout(attn_output))
        
        # Feed forward + residual connection
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        
        return x


class IAATransformer(nn.Module):
    def __init__(self, input_size, d_model, nhead, num_layers, num_classes, dropout=0.3):
        super(IAATransformer, self).__init__()
        
        self.input_projection = nn.Linear(1, d_model)
        self.layers = nn.ModuleList([
            IAATransformerLayer(d_model, nhead, dim_feedforward=512, dropout=dropout)
            for _ in range(num_layers)
        ])
        self.fc = nn.Linear(d_model, num_classes)
        self.dropout = nn.Dropout(dropout)
        self.alpha = nn.Parameter(torch.tensor(0.1))
    
    def forward(self, x, freq_bias_all):
        x = x.unsqueeze(2)
        x = self.input_projection(x)
        for layer in self.layers:
            x = layer(x)
        x = x.mean(dim=1)
        x = self.dropout(x)
        x = self.fc(x)
        x = x + torch.abs(self.alpha) * freq_bias_all
        return x

In [25]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model = IAATransformer(
    input_size=71,
    d_model=128,
    nhead=4,
    num_layers=3,
    num_classes=15,
    dropout=0.3
).to(device)

freq_bias_all = freq_bias.to(device)

print(f"Device: {device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Device: mps
Parameters: 597,008


In [26]:
#loss function
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

print("Loss: CrossEntropyLoss")
print("Optimizer: Adam, lr=0.0005")

Loss: CrossEntropyLoss
Optimizer: Adam, lr=0.0005


In [27]:
print("Training Started")
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

for epoch in range(20):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_X, freq_bias_all)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
    print(f"Epoch {epoch+1}/20 - Loss: {running_loss/len(train_loader):.4f}, Accuracy: {100*correct/total:.2f}%")

Training Started
Epoch 1/20 - Loss: 0.5544, Accuracy: 74.52%
Epoch 2/20 - Loss: 0.3039, Accuracy: 83.41%
Epoch 3/20 - Loss: 0.2574, Accuracy: 85.68%


KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), '../models/transformer.pth')

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch_X, batch_y, batch_bias in val_loader:
        batch_X = batch_X.to(device)
        batch_bias = batch_bias.to(device)
        
        outputs = model(batch_X, batch_bias)
        _, predicted = torch.max(outputs, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(batch_y.numpy())

all_preds = le.inverse_transform(all_preds)
all_labels = le.inverse_transform(all_labels)

print(classification_report(all_labels, all_preds, digits=4))

In [ ]:
for name, param in model.named_parameters():
    if 'alpha' in name:
        print(f"Learned alpha: {param.item():.4f}")